<a href="https://colab.research.google.com/github/iammustafatz/ETL-using-Luigi/blob/main/ETL_using_Luigi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Installation of Luigi

In [1]:
!pip install luigi

##Cloning the git repository to extract data

In [2]:
!git clone https://github.com/iammustafatz/ETL-using-Luigi.git

'git' n'est pas reconnu en tant que commande interne
ou externe, un programme ex�cutable ou un fichier de commandes.


##Luigi Pipeline

In [6]:
import pandas as pd
import numpy as np
import datetime as dt
import luigi
from pathlib import Path
import sqlite3
from sqlalchemy import create_engine

# Creating a SQLite database
def create_sqlite_db(db_path):
    conn = sqlite3.connect(db_path)
    conn.close()

# Creating a SQL engine
def create_sql_engine(db_path):
    return create_engine(f'sqlite:///{db_path}')

# Function to impute numerical values where null values are present
def impute_numerical_values(df):
    numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
    numerical_cols_null_counter = df[numerical_cols].isna().sum().sort_values(ascending=False)
    for col, null_count in numerical_cols_null_counter.items():
        if null_count > 0:
            median_value = np.nanmedian(df[col])
            df[col].fillna(median_value, inplace=True)

# Function to impute categorical values where null values are present
def impute_categorical_values(df):
    categorical_cols = df.select_dtypes(exclude=np.number).columns.tolist()
    categorical_cols_null_counter = df[categorical_cols].isna().sum().sort_values(ascending=False)
    for col, null_count in categorical_cols_null_counter.items():
        if null_count > 0:
            mode_value = df[col].mode()[0]
            df[col].fillna(mode_value, inplace=True)

class Extract(luigi.Task):
    date = luigi.DateParameter(default=dt.date.today())

    def output(self):
        return luigi.LocalTarget(OUTPUT_PATH / f'extracted_train_{self.date}.csv')

    def run(self):
        try:
            data = pd.read_csv(file_path)
            data.to_csv(self.output().path, index=False)
        except Exception as e:
            print(f"Error in Extract task: {e}")

class Transform(luigi.Task):
    date = luigi.DateParameter(default=dt.date.today())

    def requires(self):
        return Extract(self.date)

    def output(self):
        return luigi.LocalTarget(OUTPUT_PATH / f'transformed_train_{self.date}.csv')

    def run(self):
        try:
            extracted_data = pd.read_csv(self.input().path)
            total_nulls = extracted_data.isnull().sum().sort_values(ascending=False)
            percent_nulls = ((extracted_data.isnull().sum() / extracted_data.isnull().count()) * 100).sort_values(ascending=False)
            missing_data = pd.concat([total_nulls, percent_nulls], axis=1, keys=['Total', 'Percent'])
            cols_to_drop = missing_data[missing_data['Percent'] > 80].index.tolist()
            extracted_data.drop(cols_to_drop, axis=1, inplace=True)
            impute_numerical_values(extracted_data)
            extracted_data['MasVnrType'].fillna('None', inplace=True)
            impute_categorical_values(extracted_data)
            extracted_data.to_csv(self.output().path, index=False)
        except Exception as e:
            print(f"Error in Transform task: {e}")

class Load(luigi.Task):
    date = luigi.DateParameter(default=dt.date.today())

    def requires(self):
        return Transform(self.date)

    def output(self):
        # This is just a placeholder; you can remove it if not needed.
        return luigi.LocalTarget(OUTPUT_PATH / f'load_complete_{self.date}.txt')

    def run(self):
        try:
            transformed_data = pd.read_csv(self.input().path)
            transformed_data.to_sql('output', engine, index=False, if_exists='replace')
            with open(self.output().path, 'w') as f:
                f.write("Load task completed successfully.")
        except Exception as e:
            print(f"Error in Load task: {e}")

# File paths
file_path = 'train.csv'  # path to fetch data
OUTPUT_PATH = Path('/ETL-using-Luigi-main')  # path to store data

# Database setup
db_path = 'my.db'
create_sqlite_db(db_path)
engine = create_sql_engine(db_path)

# Building Luigi task and calling Load class
luigi.build([Load()], local_scheduler=True)

DEBUG: Checking if Load(date=2024-11-01) is complete
DEBUG: Checking if Transform(date=2024-11-01) is complete
INFO: Informed scheduler that task   Load_2024_11_01_116e987ba6   has status   PENDING
DEBUG: Checking if Extract(date=2024-11-01) is complete
INFO: Informed scheduler that task   Transform_2024_11_01_116e987ba6   has status   PENDING
INFO: Informed scheduler that task   Extract_2024_11_01_116e987ba6   has status   PENDING
INFO: Done scheduling tasks
INFO: Running Worker with 1 processes
DEBUG: Asking scheduler for work...
DEBUG: Pending tasks: 3
INFO: [pid 20048] Worker Worker(salt=6298981849, workers=1, host=LAPTOP-OMAPFQEB, username=mylen, pid=20048) running   Extract(date=2024-11-01)


INFO: [pid 20048] Worker Worker(salt=6298981849, workers=1, host=LAPTOP-OMAPFQEB, username=mylen, pid=20048) done      Extract(date=2024-11-01)
DEBUG: 1 running tasks, waiting for next task to finish
INFO: Informed scheduler that task   Extract_2024_11_01_116e987ba6   has status   DONE
DEBUG: Asking scheduler for work...
DEBUG: Pending tasks: 2
INFO: [pid 20048] Worker Worker(salt=6298981849, workers=1, host=LAPTOP-OMAPFQEB, username=mylen, pid=20048) running   Transform(date=2024-11-01)
ERROR: [pid 20048] Worker Worker(salt=6298981849, workers=1, host=LAPTOP-OMAPFQEB, username=mylen, pid=20048) failed    Transform(date=2024-11-01)
Traceback (most recent call last):
  File "c:\Users\mylen\OneDrive\Documents\Veille\ETL\.venv\Lib\site-packages\luigi\worker.py", line 195, in run
    raise RuntimeError('Unfulfilled %s at run time: %s' % (deps, ', '.join(missing)))
RuntimeError: Unfulfilled dependency at run time: Extract_2024_11_01_116e987ba6 (\ETL-using-Luigi-main\extracted_train_2024-11-

Error in Extract task: Cannot save file into a non-existent directory: '\ETL-using-Luigi-main'


False

In [7]:
OUTPUT_PATH

WindowsPath('/ETL-using-Luigi-main')

##Checking the output table is in my.db with zero-null values

In [4]:
df = pd.read_sql('output',engine)
print(df.isna().sum().to_string())

OperationalError: (sqlite3.OperationalError) near "output": syntax error
[SQL: output]
(Background on this error at: https://sqlalche.me/e/20/e3q8)